In [26]:
import pandas as pd
import ast
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 1. Load dataset
# ----------------------------
df = pd.read_csv("songs.csv")

# ----------------------------
# 2. Ensure artist column is list
# ----------------------------
def ensure_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [x]
    return x

df['artists'] = df['artists'].apply(ensure_list)

# ----------------------------
# 3. Normalize song names
# ----------------------------
df['name_clean'] = df['name'].str.strip().str.lower()

# ----------------------------
# 4. One-hot encode genre
# ----------------------------
genre_dummies = pd.get_dummies(df['genre'], prefix='genre')

# OPTIONAL: reduce genre dominance
genre_dummies = genre_dummies * 0.5

df = pd.concat([df, genre_dummies], axis=1)

# ----------------------------
# 5. Feature selection (audio + popularity + genre)
# ----------------------------
base_features = ['danceability', 'energy', 'loudness', 'speechiness',
                 'acousticness', 'instrumentalness', 'liveness',
                 'valence', 'tempo', 'popularity']

features = base_features + list(genre_dummies.columns)

X = df[features]

# ----------------------------
# 6. Scale features
# ----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ----------------------------
# 7. KMeans clustering
# ----------------------------
kmeans = KMeans(n_clusters=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# ----------------------------
# 8. Helper functions
# ----------------------------
def get_main_artist(artist_list):
    return artist_list[0] if isinstance(artist_list, list) and len(artist_list) > 0 else None

def get_featured_artists(artist_list):
    return artist_list[1:] if isinstance(artist_list, list) and len(artist_list) > 1 else []

def has_common_artist(list1, list2):
    return len(set(list1) & set(list2)) > 0

# ----------------------------
# 9. Recommendation function
# ----------------------------
def recommend_songs(song_name, artist_name, df, num_recommendations=5):
    
    # Normalize input
    song_name_clean = song_name.strip().lower()
    artist_name_clean = artist_name.strip().lower()
    
    # Find song
    filtered = df[
        (df['name_clean'] == song_name_clean) &
        (df['artists'].apply(lambda x: (get_main_artist(x) or "").lower() == artist_name_clean))
    ]
    
    if filtered.empty:
        print(f"❌ Song '{song_name}' by '{artist_name}' not found")
        return []
    
    # Extract row
    row = filtered.iloc[0]
    
    song_cluster = row['Cluster']
    song_popularity = row['popularity']
    input_featured = get_featured_artists(row['artists'])
    input_main_artist = get_main_artist(row['artists'])
    input_genre = row['genre']
    
    # Same cluster
    same_cluster = df[df['Cluster'] == song_cluster].copy()
    
    # Remove original song
    same_cluster = same_cluster[
        ~(
            (same_cluster['name_clean'] == song_name_clean) &
            (same_cluster['artists'].apply(lambda x: (get_main_artist(x) or "").lower() == artist_name_clean))
        )
    ]
    
    # ----------------------------
    # 🔥 Scoring
    # ----------------------------
    
    # Popularity similarity
    same_cluster['popularity_diff'] = abs(
        same_cluster['popularity'] - song_popularity
    )
    
    # Same artist penalty
    same_cluster['artist_penalty'] = same_cluster['artists'].apply(
        lambda x: get_main_artist(x) == input_main_artist
    ).astype(int)
    
    # Collaborator bonus
    same_cluster['collab_bonus'] = same_cluster['artists'].apply(
        lambda x: has_common_artist(get_featured_artists(x), input_featured)
    ).astype(int)
    
    # Genre match bonus
    same_cluster['genre_match'] = (
        same_cluster['genre'] == input_genre
    ).astype(int)
    
    # Final score (lower = better)
    same_cluster['score'] = (
        same_cluster['popularity_diff'] * 0.6 +
        same_cluster['artist_penalty'] * 5 -
        same_cluster['collab_bonus'] * 2 -
        same_cluster['genre_match'] * 2
    )
    
    # Sort and return
    recommendations = same_cluster.sort_values(by='score').head(num_recommendations)
    
    return recommendations[['name', 'artists', 'genre', 'popularity']]

# ----------------------------
# 10. Test
# ----------------------------
input_song = "DNA."
input_artist = "Kendrick Lamar"

recommended = recommend_songs(input_song, input_artist, df, 5)

print(f"\n🎵 Songs similar to '{input_song}' by {input_artist}:\n")
print(recommended)


🎵 Songs similar to 'DNA.' by Kendrick Lamar:

                                                     name  \
15814                                            Toxicity   
8808                                      Paint It, Black   
18241                                    I Wanna Be Yours   
124774                                  Courage to Change   
18521   Earned It (Fifty Shades Of Grey) - From The "F...   

                     artists     genre  popularity  
15814     [System Of A Down]  Rock/Pop          78  
8808    [The Rolling Stones]  Rock/Pop          78  
18241       [Arctic Monkeys]  Rock/Pop          78  
124774                 [Sia]  Rock/Pop          78  
18521           [The Weeknd]  Rock/Pop          78  


In [9]:
import pandas as pd
import ast
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 1. LOAD DATASET
# ----------------------------
df = pd.read_csv("songs.csv")

# ----------------------------
# 2. FIX ARTIST COLUMN
# ----------------------------
def ensure_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return [x]
    return x

df['artists'] = df['artists'].apply(ensure_list)

# ----------------------------
# 3. NORMALIZE
# ----------------------------
df['name_clean'] = df['name'].str.strip().str.lower()

# ----------------------------
# 4. FEATURES
# ----------------------------
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness',
            'valence', 'tempo', 'popularity']

X = df[features]

# ----------------------------
# 5. SCALE + CLUSTER
# ----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# ----------------------------
# 6. MATCH INPUT PLAYLIST
# ----------------------------
def match_songs(input_songs, df):
    """
    input_songs = list of (song_name, artist_name)
    """
    matched_indices = []

    for song_name, artist_name in input_songs:
        song_name = song_name.strip().lower()
        artist_name = artist_name.strip().lower()

        match = df[
            (df['name_clean'] == song_name) &
            (df['artists'].apply(lambda x: (x[0] if x else "").lower() == artist_name))
        ]

        if not match.empty:
            matched_indices.append(match.index[0])

    return matched_indices

# ----------------------------
# 7. GET DOMINANT CLUSTER
# ----------------------------
def get_playlist_cluster(indices, df):
    clusters = df.loc[indices, 'Cluster']
    return clusters.mode()[0]

# ----------------------------
# 8. RECOMMEND
# ----------------------------
def recommend_from_playlist(input_songs, df, n=20):

    indices = match_songs(input_songs, df)

    if not indices:
        print("❌ No songs matched in dataset")
        return None

    cluster = get_playlist_cluster(indices, df)

    # exclude input songs
    input_ids = set(df.loc[indices, 'id'])

    candidates = df[
        (df['Cluster'] == cluster) &
        (~df['id'].isin(input_ids))
    ]

    recommendations = candidates.sample(n=min(n, len(candidates)))

    return recommendations[['id', 'name', 'artists', 'genre', 'popularity']]

# ----------------------------
# 9. TEST INPUT
# ----------------------------
input_playlist = [
    ("DNA.", "Kendrick Lamar"),
    ("HUMBLE.", "Kendrick Lamar"),
    ("A.D.H.D","Kendrick Lamar"),
    ("Rigamortis","Kendrick Lamar"),
    ("SICKO MODE", "Travis Scott"),
    
    
]

recommended = recommend_from_playlist(input_playlist, df, n=10)

print("\n🎵 Recommended Playlist:\n")
print(recommended)


🎵 Recommended Playlist:

                            id  \
117931  0McGrsz2jWeeU0YmKaY0dQ   
108832  5wMh9hhqKQ4JfqLuvqAC7s   
11248   0cO7JEo8deKuQMWpDyjenY   
164691  2YseT4eAQUDdNnxZVYUATG   
137894  2F8mhprAH20PKf32agtk1d   
49065   5VMTSLp2WX3dsnsVWiVGB7   
130156  3Rmz8BYMGWFmDOSPREEmnW   
71715   65V4v8u4fi7TfnNdyBFzvX   
14346   33HRECrmuelZxOpid6XTNX   
66363   1OVSyiY9YkHOq1FzEb94sq   

                                                     name  \
117931                                         Temptation   
108832                            Cold (feat. Tory Lanez)   
11248   Time for Me to Fly - As heard in the Netflix s...   
164691                                     Drop That Bass   
137894                                   Behind Blue Eyes   
49065                                Mind - 2005 Remaster   
130156         Little Miss Queen of Darkness - Stereo Mix   
71715                                       Un Nuevo Amor   
14346                                             